In [63]:
from cobra import Model, Reaction, Metabolite

In [64]:
model = Model('NRRL_1')

In [65]:
model

Name,NRRL_1
Memory address,26b5d0d0cb0
Number of metabolites,0
Number of reactions,0
Number of genes,0
Number of groups,0
Objective expression,0
Compartments,


---------------------------------------------------------------------------------------------------------------------------------------

In [66]:
from cobra import Metabolite
import pandas as pd
from tqdm import tqdm
import pickle

metabolites = {}
metabolites_df = pd.read_csv("../data/NRRL_1/7_Annotation/chr.kegg.metabolites_backup.csv")

for _, row in tqdm(metabolites_df.iterrows(), 
                       total=len(metabolites_df),
                       desc="Adding metabolites"):
    metabolite_id = row["Metabolite ID"]
    formula = row["Formula/Composition"]
    metabolite_name = row["Name"]
    compartment = row["Compartment"]
    
    if not any(metabolite_id.endswith(suffix) for suffix in ["_c", "_e"]):
        metabolite_id = metabolite_id + "_c"
    
    metabolite = Metabolite(
        id=metabolite_id,
        formula=formula,
        name=metabolite_name,
        compartment=compartment
    )
    metabolites[metabolite_id] = metabolite
    # Add the metabolite to the model
    model.add_metabolites([metabolite])
    
with open("../data/NRRL_1/7_Annotation/metabolites.pkl", "wb") as f:
    pickle.dump(metabolites, f)
    print("Metabolites saved to ../data/NRRL_1/7_Annotation/metabolites.pkl")

Adding metabolites: 100%|████████████████████████████████████████████████████████| 1793/1793 [00:00<00:00, 9319.08it/s]

Metabolites saved to ../data/NRRL_1/7_Annotation/metabolites.pkl


In [67]:
model.metabolites

[<Metabolite C00345_c at 0x26c1194ea30>,
 <Metabolite C00006_c at 0x26c1194eb70>,
 <Metabolite C00199_c at 0x26c1194e850>,
 <Metabolite C00011_c at 0x26c1194e7b0>,
 <Metabolite C00005_c at 0x26c1194e710>,
 <Metabolite C00080_c at 0x26c1194e670>,
 <Metabolite C00003_c at 0x26c1194e5d0>,
 <Metabolite C00004_c at 0x26c1194e530>,
 <Metabolite C00251_c at 0x26c1194e490>,
 <Metabolite C00014_c at 0x26c1194e3f0>,
 <Metabolite C00108_c at 0x26c1194e350>,
 <Metabolite C00022_c at 0x26c1194e2b0>,
 <Metabolite C00001_c at 0x26c1194e210>,
 <Metabolite C00064_c at 0x26c1194e170>,
 <Metabolite C00025_c at 0x26c1194e0d0>,
 <Metabolite C05898_c at 0x26c1194e030>,
 <Metabolite C11827_c at 0x26c1194df90>,
 <Metabolite C04574_c at 0x26c1194def0>,
 <Metabolite G10555_c at 0x26c1194de50>,
 <Metabolite G10557_c at 0x26c1194ddb0>,
 <Metabolite C00029_c at 0x26c1194dd10>,
 <Metabolite C00167_c at 0x26c1194dc70>,
 <Metabolite C00051_c at 0x26c1194dbd0>,
 <Metabolite C01419_c at 0x26c1194db30>,
 <Metabolite C00

In [68]:
model

Name,NRRL_1
Memory address,26b5d0d0cb0
Number of metabolites,1792
Number of reactions,0
Number of genes,0
Number of groups,0
Objective expression,0
Compartments,"Cytosol, e"


In [69]:
model.metabolites.get_by_id('C_Spinosad_c')

Metabolite identifier,C_Spinosad_c
Name,Spinosad
Memory address,0x26ccffe54f0
Formula,Spinosad
Compartment,Cytosol
In 0 reaction(s),


In [70]:
model.metabolites.get_by_id('C_Spinosad_e')

Metabolite identifier,C_Spinosad_e
Name,Spinosad
Memory address,0x26ccffe5590
Formula,Spinosad
Compartment,e
In 0 reaction(s),


In [19]:
reaction_short = '2 C00002 + C00064 + C00288 + C00001 <=> 2 C00008 + C00009 + C00025 + C00169'
reactants, products = reaction_short.split(" <=> ")
for reactant in reactants.split("+"):
    reactant = reactant.strip()
    if ' ' in reactant:
        coeff, metabolite_id = reactant.split(' ', 1)
        coeff = float(coeff) * -1  # 反应物系数为负
    else:
        coeff, metabolite_id = -1, reactant  # 默认系数为-1
    if not any(metabolite_id.endswith(suffix) for suffix in ["_c", "_e"]):
        metabolite_id += "_c"
    metabolite = metabolites[metabolite_id]
    print(f"反应物: {metabolite_id}, 系数: {coeff}", metabolite)

for product in products.split("+"):
    product = product.strip()
    if ' ' in product:
        coeff, metabolite_id = product.split(' ', 1)
        coeff = float(coeff)  # 产物系数为正
    else:
        coeff, metabolite_id = 1, product  # 默认系数为1
    if not any(metabolite_id.endswith(suffix) for suffix in ["_c", "_e"]):
        metabolite_id += "_c"
    metabolite = metabolites[metabolite_id]
    print(f"产物: {metabolite_id}, 系数: {coeff}", metabolite)

反应物: C00002_c, 系数: -2.0 C00002_c
反应物: C00064_c, 系数: -1 C00064_c
反应物: C00288_c, 系数: -1 C00288_c
反应物: C00001_c, 系数: -1 C00001_c
产物: C00008_c, 系数: 2.0 C00008_c
产物: C00009_c, 系数: 1 C00009_c
产物: C00025_c, 系数: 1 C00025_c
产物: C00169_c, 系数: 1 C00169_c


In [71]:
from cobra import Reaction
from cobra.io import save_json_model
import pickle
import pandas as pd
from tqdm import tqdm
import os
from cobra import Configuration
Configuration().solver = "glpk"

from datetime import datetime

starttime = datetime.now()

METABOLITE_FILE = "../data/NRRL_1/7_Annotation/metabolites.pkl"

if os.path.exists(METABOLITE_FILE):
    with open(METABOLITE_FILE, "rb") as f:
        metabolites = pickle.load(f)
    print(f"模型包含 {len(metabolites)} 个代谢物")
else:
    raise FileNotFoundError(f"文件 {METABOLITE_FILE} 不存在，请先运行添加代谢物的代码块")

reactions_df = pd.read_csv("../data/NRRL_1/7_Annotation/chr.kegg.reactions_with_genes_backup.csv",
                           dtype={'Genes': str}).fillna({'Genes': ''})

# 添加反应
for _, row in tqdm(reactions_df.iterrows(), total=len(reactions_df), desc="Adding reactions"):
    try:
        reaction_id = row["Reaction ID"]
        reaction_name = row["Reaction (Long)"]
        # reaction_subsystem = row["Subsystem"]
        # 判断反应是否可逆
        if row["Reversible"]:
            lower_bound = -1000.0
            upper_bound = 1000.0
        else:
            lower_bound = 0.0
            upper_bound = 1000.0
        # 创建反应对象
        reaction = Reaction(
            id=reaction_id,
            name=reaction_name,
            # subsystem=reaction_subsystem,
            lower_bound=lower_bound,
            upper_bound=upper_bound
        )
        # 添加反应物和产物
        reaction_short = row["Reaction (Short)"]
        # 2 C00027 <=> C00007 + 2 C00001 形如这样的反应 <=> 前面赋值负，后面赋值正
        reactants, products = reaction_short.split(" <=> ")
        # 处理反应物
        for reactant in reactants.split("+"):
            reactant = reactant.strip()
            # 解析系数和代谢物ID
            if ' ' in reactant:
                coeff, metabolite_id = reactant.split(' ', 1)
                coeff = float(coeff) * -1  # 反应物系数为负
            else:
                coeff, metabolite_id = -1, reactant  # 默认系数为-1
            
            # 如果metabolite_id不以"_c"或"_e"结尾，则添加"_c"
            if not any(metabolite_id.endswith(suffix) for suffix in ["_c", "_e"]):
                metabolite_id += "_c"
            metabolite = metabolites[metabolite_id]

            # Adding metabolites to a reaction uses a dictionary of the metabolites and their stoichiometric coefficients.
            # A group of metabolites can be added all at once, or they can be added one at a time.
            reaction.add_metabolites({metabolite: coeff})

        # 处理产物
        for product in products.split("+"):
            product = product.strip()
            # 解析系数和代谢物ID
            if ' ' in product:
                coeff, metabolite_id = product.split(' ', 1)
                coeff = float(coeff)  # 产物系数为正
            else:
                coeff, metabolite_id = 1, product  # 默认系数为1
                
            # 如果metabolite_id不以"_c"或"_e"结尾，则添加"_c"
            if not any(metabolite_id.endswith(suffix) for suffix in ["_c", "_e"]):
                metabolite_id += "_c"
            metabolite = metabolites[metabolite_id]
            
            reaction.add_metabolites({metabolite: coeff})

        # 添加反应规则
        reaction.gene_reaction_rule = row["Genes"]

        model.add_reactions([reaction])
    except Exception as e:
        print(f"处理反应 {reaction_id} 时出错: {str(e)}")
        continue

endtime = datetime.now()
print(endtime - starttime)

模型包含 1792 个代谢物


Adding reactions: 100%|████████████████████████████████████████████████████████████| 1627/1627 [02:59<00:00,  9.06it/s]

0:02:59.502602


In [73]:
for metabolite in tqdm(model.metabolites, desc="Adding boundary reactions"):
    if metabolite.id.endswith("_e"):
        print(f"Adding boundary reaction for {metabolite.id}")
        model.add_boundary(metabolite, type="exchange")
# metabolite：C_Spinosad_e
# exchange reaction：EX_C_Spinosad_e

Adding boundary reactions: 100%|█████████████████████████████████████████████████| 1792/1792 [00:00<00:00, 7332.00it/s]

Adding boundary reaction for C_Spinosad_e
Adding boundary reaction for C00245_e
Adding boundary reaction for C06232_e
Adding boundary reaction for C14819_e
Adding boundary reaction for C00378_e
Adding boundary reaction for C00719_e
Adding boundary reaction for C00114_e
Adding boundary reaction for C00059_e
Adding boundary reaction for C00244_e
Adding boundary reaction for C00208_e
Adding boundary reaction for C00794_e
Adding boundary reaction for C00185_e
Adding boundary reaction for C00121_e
Adding boundary reaction for C00025_e
Adding boundary reaction for C00012_e
Adding boundary reaction for C00175_e
Adding boundary reaction for C00140_e
Adding boundary reaction for C00095_e
Adding boundary reaction for C00014_e
Adding boundary reaction for C00001_e
Adding boundary reaction for C00007_e
Adding boundary reaction for C00011_e
Adding boundary reaction for C00058_e
Adding boundary reaction for C02323_e
Adding boundary reaction for C00160_e
Adding boundary reaction for C00084_e
Adding b

In [74]:
model.metabolites.get_by_id("C_Spinosad_e")

Metabolite identifier,C_Spinosad_e
Name,Spinosad
Memory address,0x26ccffe5590
Formula,Spinosad
Compartment,e
In 2 reaction(s),"EX_C_Spinosad_e, R_Spinosad"


In [75]:
def set_production_objective(model, target_metabolite):
    """设置目标代谢物的最大化生产"""
    # 重置所有目标
    model.objective = {}

    # 设置新的目标
    target_metabolite_id = target_metabolite + "_e"
    target_reaction_id = "EX_" + target_metabolite + "_e"
    if model.metabolites.get_by_id(target_metabolite_id):
        export_reaction = model.reactions.get_by_id(target_reaction_id)
        model.objective = export_reaction
        print(f"目标设置为: {target_metabolite} 的最大化生产")
    else:
        raise ValueError(f"未找到目标代谢物: {target_metabolite}")

# R_Spinosad	Spinosad = Spinosad_e	C_Spinosad <=> C_Spinosad_e
# EX_C_Spinosad_e                       C_Spinosad_e <=>

# 设置目标产物
set_production_objective(model, 'C_Spinosad')

目标设置为: C_Spinosad 的最大化生产


In [76]:
model.metabolites.get_by_id("C_Spinosad_c")

Metabolite identifier,C_Spinosad_c
Name,Spinosad
Memory address,0x26ccffe54f0
Formula,Spinosad
Compartment,Cytosol
In 2 reaction(s),"R_Spinosad, R_Assemble"


In [77]:
model.metabolites.get_by_id("C_Spinosad_e")

Metabolite identifier,C_Spinosad_e
Name,Spinosad
Memory address,0x26ccffe5590
Formula,Spinosad
Compartment,e
In 2 reaction(s),"EX_C_Spinosad_e, R_Spinosad"


In [78]:
solution = model.optimize()

In [79]:
print("\n优化结果:")
print(f"目标值: {solution.objective_value:.6f}")
print(f"求解状态: {solution.status}")

# 了解Spinosad的输入输出行为
print("\nSpinosad生产情况:")
print(model.metabolites.C_Spinosad_c.summary())
print(model.metabolites.C_Spinosad_e.summary())

# 了解主要的能量(C00002<->atp)生产和消耗反应
print(model.metabolites.C00002_c.summary())

# 输出主要通量
print("\n主要代谢物通量:")
for ex in model.exchanges:
    if abs(solution.fluxes[ex.id]) > 1e-6:
        print(f"{ex.name}: {solution.fluxes[ex.id]:.6f}")


优化结果:
目标值: 90.909091
求解状态: optimal

Spinosad生产情况:
C_Spinosad_c
Formula: Spinosad

Producing Reactions
-------------------
Percent  Flux   Reaction                                                                      Definition
100.00% 90.91 R_Assemble 3.0 C00019_c + C02199_c + C18034_c + C_AGL_c --> C00015_c + 3.0 C00021_c + C...

Consuming Reactions
-------------------
Percent   Flux   Reaction                    Definition
100.00% -90.91 R_Spinosad C_Spinosad_c --> C_Spinosad_e
C_Spinosad_e
Formula: Spinosad

Producing Reactions
-------------------
Percent  Flux   Reaction                    Definition
100.00% 90.91 R_Spinosad C_Spinosad_c --> C_Spinosad_e

Consuming Reactions
-------------------
Percent   Flux        Reaction        Definition
100.00% -90.91 EX_C_Spinosad_e C_Spinosad_e <=> 
C00002_c
Formula: C10H16N5O13P3

Producing Reactions
-------------------
Percent  Flux Reaction                                                        Definition
 15.43%   814   R00087         

In [81]:
# 检查反应的上下限
nan_reactions = [
    r.id for r in model.reactions 
    if pd.isna(r.lower_bound) or pd.isna(r.upper_bound)
]
print("含NaN边界条件的反应:", nan_reactions)

# 检查代谢物的注释字段（如电荷、化学式）
nan_metabolites = [
    m.id for m in model.metabolites 
    if any(pd.isna(val) for val in m.notes.values())
]
print("含NaN注释的代谢物:", nan_metabolites)

# 检查模型的全局参数（如目标函数系数）
if pd.isna(model.solver.objective.expression):
    print("目标函数存在NaN")

含NaN边界条件的反应: []
含NaN注释的代谢物: []


In [82]:
from cobra.io import write_sbml_model

write_sbml_model(model, f"../model/{model.id}.xml")

TypeError: in method 'FbcSpeciesPlugin_setChemicalFormula', argument 2 of type 'std::string const &'

In [80]:
save_json_model(model, f"../model/{model.id}.json")

ValueError: Out of range float values are not JSON compliant: nan

In [83]:
# 检查无效的化学式字段
invalid_metabolites = []
for met in model.metabolites:
    if not isinstance(met.formula, str):
        invalid_metabolites.append(met.id)
        met.formula = ''  # 强制设为空字符串

if invalid_metabolites:
    print(f"修复以下代谢物的化学式类型: {invalid_metabolites}")

修复以下代谢物的化学式类型: ['C00030_c', 'C00028_c', 'C01647_c', 'C00999_c', 'C00996_c', 'C00138_c', 'C00139_c', 'C15603_c', 'C15602_c', 'C03024_c', 'C03161_c', 'C22154_c', 'C22150_c', 'C22155_c', 'C22151_c', 'C04253_c', 'C04570_c', 'C15805_c', 'C15806_c', 'C15804_c', 'C15807_c', 'C17023_c', 'C05359_c', 'C00662_c', 'C00667_c', 'C00340_c', 'C00435_c', 'C03688_c', 'C19645_c', 'C19646_c', 'C02869_c', 'C02745_c', 'C01641_c', 'C01629_c']


In [84]:
save_json_model(model, f"../model/{model.id}.json")

问题可能出在：没有完全吸收tju模型的反应；_e物质少，exchange反应基本全加上去了，几个没有的删了；add_boundary；反应的可逆性；生物量

gene_info

In [85]:
from cobra.flux_analysis import (single_gene_deletion, single_reaction_deletion)

single_reaction_deletion_results = single_reaction_deletion(model)
single_reaction_deletion_results.to_csv('../data/NRRL_1/7_Annotation/single_reaction_deletion_results.csv')
print(f"结果已保存到: '../data/single_reaction_deletion_results.csv'")

结果已保存到: '../data/single_reaction_deletion_results.csv'


In [88]:
def find_essential_single_reaction(model, production_threshold=0.5):
    """
    Find essential single reaction
    param production_threshold: The minimum production change threshold to consider a reaction essential.
    return: A list of essential reactions.
    """
    essential_reactions = []
    objective_value = model.optimize().objective_value
    single_reaction_deletion_results = pd.read_csv('../data/NRRL_1/7_Annotation/single_reaction_deletion_results.csv')
    for _, row in single_reaction_deletion_results.iterrows():
        if row['status'] != 'optimal':
            continue
        if abs((row['growth'] - objective_value) / objective_value) >= production_threshold:
            essential_reactions.append(row['ids'])
    return essential_reactions

essential_reactions = find_essential_single_reaction(model)
print(f"Essential reactions: {essential_reactions}")

Essential reactions: ["{'R03018'}", "{'R00130'}", "{'R06513'}", "{'R00859'}", "{'R04231'}", "{'R00293'}", "{'R_Spinosad'}", "{'R01226'}", "{'R06430'}", "{'R08705'}", "{'R_Sulfate'}", "{'R02328'}", "{'R08930'}", "{'R08704'}", "{'R_Assemble'}", "{'R08932'}", "{'R02473'}", "{'R08931'}", "{'R03269'}", "{'R02472'}", "{'EX_C00059_e'}", "{'EX_C_Spinosad_e'}", "{'R06428'}", "{'R_AGL'}"]


R06430不是R06434